# Colab Benchmark Profiler (Multi-Producer Stream & 8 Consumer Workers)

This benchmark tests pipeline execution with **4 Parallel Download Producer Threads** and **8 Consumer Workers** (`max_queue_size = 16`).

### Performance Strategy:
1. **4 Parallel S3 Download Producers**: Streams downloads concurrently to eliminate single-downloader bottleneck.
2. **8 CPU Consumer Threads**: Parallel text extraction and entity parsing.
3. **Network Download Speed Tracking**: Measures transfer throughput (Mbps & MB/s).

## Step 1: Install Dependencies

In [ ]:
!pip install -q pymupdf pyarrow duckdb rich spacy requests urllib3 matplotlib psutil

## Step 2: Auto-Load Pipeline Code & Modules

In [ ]:
import os, sys
repo_dir = "/content/aws_indian_judgements"
if os.path.exists("/content") and not os.path.exists(repo_dir):
    print("Cloning repository into Colab environment...")
    !git clone https://github.com/duttadev/aws_indian_judgements.git {repo_dir}
%cd /content/aws_indian_judgements
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

print(f"Working Directory: {os.getcwd()}")
from config import PipelineConfig
from pipeline.storage import StorageManager
from batch_runner import BatchScheduler
from pipeline.cleaner import TextCleaner
from pipeline.decoupled_runner import DecoupledPipelineScheduler
print("✓ Pipeline modules loaded successfully!")

## Step 3: Fetch Real English S3 Judgment Keys

In [ ]:
import time, json, glob, requests
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib.pyplot as plt
from pipeline.cleaner import TextCleaner

def get_real_english_s3_keys(max_keys=100):
    raw_keys = []
    for kfile in ["./data/hc_keys_sample.json", "./data/sc_keys_sample.json"]:
        if os.path.exists(kfile):
            with open(kfile, "r") as f:
                raw_keys = json.load(f)
            break
    if not raw_keys:
        try:
            r = requests.get("https://indian-high-court-judgments.s3.amazonaws.com/?max-keys=200", timeout=10)
            root = ET.fromstring(r.text)
            ns = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
            raw_keys = [elem.find("s3:Key", ns).text for elem in root.findall("s3:Contents", ns) if elem.find("s3:Key", ns).text.endswith(".pdf")]
        except Exception as e:
            print(f"Error fetching keys: {e}")
            return []
    
    english_keys = [k for k in raw_keys if TextCleaner.is_english_key(k)]
    return english_keys[:max_keys]

benchmark_keys = get_real_english_s3_keys(max_keys=100)
print(f"Loaded {len(benchmark_keys)} English S3 PDF benchmark keys.")

## Step 4: Run Multi-Producer Streaming Benchmark (8 Consumer Workers)

In [ ]:
workers = 8
print(f"==================================================")
print(f" Multi-Producer Benchmark with max_workers = {workers}")
print(f" Sample Size: {len(benchmark_keys)} English PDFs")
print(f"==================================================")

scratch_path = f"/tmp/benchmark_workers_8"
os.makedirs(scratch_path, exist_ok=True)

config = PipelineConfig(
    run_id="bench-multi-prod-8",
    base_output_dir=scratch_path,
    local_scratch_dir=scratch_path,
    s3_base_url="https://indian-high-court-judgments.s3.amazonaws.com",
    batch_size=50,
    max_workers=8,
    keep_pdf_files=False,
    keep_intermediate_artifacts=True,
    resume_enabled=False
)

storage = StorageManager(config)
scheduler = DecoupledPipelineScheduler(config, storage, max_queue_size=16)

t0 = time.time()
summary = scheduler.run_decoupled_pipeline(benchmark_keys)
elapsed = time.time() - t0

pdf_tp = summary.get('throughput_pdfs_per_sec', 0)
page_tp = summary.get('throughput_pages_per_sec', 0)
ram_peak = summary.get('peak_ram_mb', 0)

print(f"\n============================================================")
print(f" MULTI-PRODUCER BENCHMARK SUMMARY:")
print(f" • Execution Time        : {elapsed:.2f} seconds")
print(f" • Document Throughput   : {pdf_tp:.2f} PDFs/sec")
print(f" • Page Throughput       : {page_tp:.2f} pages/sec")
print(f" • Peak RAM RSS Memory   : {ram_peak} MB")
print(f"============================================================")